In [ ]:
!pip install transformers datasets torch scikit-learn gradio accelerate -q

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import gradio as gr

In [ ]:
dataset = load_dataset('ag_news')
print(dataset)
print(dataset['train'][0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

small_train = dataset['train'].select(range(2000))
small_test = dataset['test'].select(range(500))

train_tokenized = small_train.map(tokenize, batched=True)
test_tokenized = small_test.map(tokenize, batched=True)

train_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

In [ ]:
import transformers

major_version = int(transformers.__version__.split('.')[1])

if major_version >= 41:
    args = TrainingArguments(
        output_dir='./bert_news',
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_steps=50,
        report_to='none'
    )
else:
    args = TrainingArguments(
        output_dir='./bert_news',
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_steps=50,
        report_to='none'
    )

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print('Accuracy:', round(results['eval_accuracy'], 4))
print('F1 Score:', round(results['eval_f1'], 4))

In [ ]:
model.save_pretrained('./bert_news_model')
tokenizer.save_pretrained('./bert_news_model')

label_map = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Science/Tech'}

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return label_map[pred]

iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label='Enter News Headline'),
    outputs=gr.Textbox(label='Predicted Topic'),
    title='News Topic Classifier'
)

iface.launch(share=True)